# ESUS reproducibility: 66 prompts

Re-extracts `prompts/validation/prompt_01.txt` … `prompt_60.txt` plus the six
initial prompts (`hello`, `cli_tool`, `expenses`, `gui`, `inventory`, `library`)
and records timing, token use, and the three-way outcome:

- **successful** — generatable with no blockers or ties;
- **review** — only reviewable blockers or open ties remain;
- **failed** — any hard blocker, crash, or timeout (`not_generatable`).

Inputs expected under `/kaggle/input`:

- `esus-code-bundle/esus-code.bundle`: a zip snapshot of this repository (including both prompt roots);
- `qwen3-4b-instruct/Qwen3-4B-Instruct-2507-Q4_K_M.gguf`: the Qwen model.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time, urllib.request, zipfile

INPUT = Path('/kaggle/input')
print('input tree:', [str(p) for p in sorted(INPUT.rglob('*'))[:30]])

def find_dir(name):
    return next((p for p in INPUT.rglob(name) if p.is_dir()), None)

def find_file(name):
    return next((p for p in INPUT.rglob(name) if p.is_file()), None)

CODE_ARCHIVE = find_file('esus-code.bundle') or find_file('esus-code.zip')
MODEL = find_file('Qwen3-4B-Instruct-2507-Q4_K_M.gguf')
WORK = Path('/kaggle/working/esus')
RUN_ID = time.strftime('repro66-%Y%m%d-%H%M%S')

assert CODE_ARCHIVE is not None, 'missing esus-code.bundle in /kaggle/input'
assert MODEL is not None, 'missing Qwen GGUF in /kaggle/input'

if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)
with zipfile.ZipFile(CODE_ARCHIVE) as archive:
    archive.extractall(WORK)
os.chdir(WORK)
print('repo:', WORK)
print('model:', MODEL)
print('run id:', RUN_ID)


In [ ]:
# Prefer the official CUDA prebuilt; building llama.cpp from source is the
# fallback because Kaggle's CUDA 13 CMake target set is inconsistent.
server_bin = shutil.which('llama-server')
server_env = dict(os.environ)
if not server_bin:
    import tarfile, urllib.request
    release = 'b12110'
    base = 'https://github.com/ggml-org/llama.cpp/releases/download'
    root = Path('/kaggle/working/llama-prebuilt')
    root.mkdir(parents=True, exist_ok=True)
    assets = [
        f'llama-{release}-bin-ubuntu-cuda-12.8-x64.tar.gz',
        f'cudart-llama-{release}-bin-ubuntu-cuda-12.8-x64.tar.gz',
    ]
    for asset in assets:
        target = root / asset
        if not target.exists():
            print('downloading', asset, flush=True)
            urllib.request.urlretrieve(f'{base}/{release}/{asset}', target)
        with tarfile.open(target) as archive:
            archive.extractall(root)
    candidates = sorted(root.rglob('llama-server'))
    if not candidates:
        # Last-resort CPU build (slow, but keeps the validation run alive).
        build_dir = Path('/kaggle/working/llama.cpp')
        if not (build_dir / '.git').is_dir():
            subprocess.run(['git', 'clone', '--depth', '1',
                            'https://github.com/ggml-org/llama.cpp', str(build_dir)], check=True)
        subprocess.run(['cmake', '-S', str(build_dir), '-B', str(build_dir / 'build'),
                        '-DGGML_CUDA=OFF', '-DLLAMA_CURL=OFF'], check=True)
        subprocess.run(['cmake', '--build', str(build_dir / 'build'),
                        '--config', 'Release', '-j2'], check=True)
        candidates = sorted((build_dir / 'build').rglob('llama-server'))
    assert candidates, 'llama-server binary not found'
    server_bin = str(candidates[0])
    lib_dirs = sorted({str(path.parent) for path in root.rglob('*.so*')})
    if lib_dirs:
        server_env['LD_LIBRARY_PATH'] = ':'.join(lib_dirs) + ':' + server_env.get('LD_LIBRARY_PATH', '')
assert Path(server_bin).is_file(), f'llama-server not found: {server_bin}'
print('llama-server:', server_bin)
print('LD_LIBRARY_PATH:', server_env.get('LD_LIBRARY_PATH', ''))


In [ ]:
# One llama-server, two decode slots sharing the loaded model.
gpu_list = []
if shutil.which('nvidia-smi'):
    listing = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True, check=False).stdout
    gpu_list = [line for line in listing.splitlines() if line.startswith('GPU ')]
print('visible GPUs:', len(gpu_list))
log_path = Path('/kaggle/working/llama-server.log')
log = open(log_path, 'w')
env = dict(server_env)
env['CUDA_VISIBLE_DEVICES'] = '0'
args = [server_bin, '-m', str(MODEL), '-c', '32768', '-ngl', '99',
        '-fa', 'on', '-b', '1024', '-ub', '1024', '--parallel', '2',
        '--port', '8080']
print('starting:', ' '.join(args), flush=True)
server = subprocess.Popen(args, stdout=log, stderr=subprocess.STDOUT, env=env)
healthy = False
for _ in range(1800):
    if server.poll() is not None:
        log.flush()
        tail = log_path.read_text(errors='replace')[-4000:] if log_path.exists() else ''
        raise RuntimeError(f'llama-server exited with {server.returncode}; log tail:\n{tail}')
    try:
        with urllib.request.urlopen('http://127.0.0.1:8080/health', timeout=2) as response:
            if response.status == 200:
                healthy = True
                break
    except Exception:
        time.sleep(1)
if not healthy:
    server.terminate()
    raise RuntimeError('llama-server did not become healthy')
print('llama-server ready, pid', server.pid, 'slots=2')


In [ ]:
run_dir = WORK / 'out' / 'validation' / 'runs' / RUN_ID
cmd = [sys.executable, 'scripts/run_validation.py',
       '--prompts', 'prompts/validation',
       '--prompts', 'prompts/initial',
       '--out', str(run_dir.parent),
       '--run-id', RUN_ID,
       '--server-url', 'http://127.0.0.1:8080',
       '--concurrency', '2',
       '--timeout', '1800']
print(' '.join(cmd), flush=True)
subprocess.run(cmd, cwd=WORK, check=True)
print('done:', run_dir)


In [ ]:
import csv, json

summary_json = run_dir / 'summary.json'
payload = json.loads(summary_json.read_text(encoding='utf-8'))
outcomes = payload['manifest'].get('outcome_counts', {})
print('TOTAL:', payload['manifest'].get('total', len(payload['results'])))
for key in ('successful', 'review', 'failed'):
    print(key.upper() + ':', outcomes.get(key, 0))

summary_csv = run_dir / 'summary.csv'
with summary_csv.open(newline='', encoding='utf-8') as handle:
    rows = list(csv.DictReader(handle))
print('\nper prompt:')
for row in rows:
    print(row['prompt'], row['outcome'], row['classification'],
          'blockers=' + row['blockers'], 'ties=' + row['open_ties'],
          'seconds=' + row['seconds'])

report = Path('/kaggle/working/repro66-report.json')
report.write_text(json.dumps({
    'run_id': RUN_ID,
    'total': payload['manifest'].get('total', len(payload['results'])),
    'outcome_counts': outcomes,
    'classification_counts': payload['manifest'].get('counts', {}),
    'results': [{key: row.get(key) for key in (
        'prompt', 'outcome', 'classification', 'blockers', 'open_ties',
        'rules', 'seconds', 'tokens_out', 'tokens_prompt')}
        for row in rows],
}, indent=2) + '\n', encoding='utf-8')
archive = shutil.make_archive(f'/kaggle/working/repro66-{RUN_ID}', 'zip', run_dir)
print('\nreport:', report)
print('archive:', archive)


## Notes

`scripts/run_validation.py` skips prompt directories with an existing
`metrics.json` unless `--force` is used. To resume in a later Kaggle session,
copy the previous `out/validation/runs/<run-id>` directory into the notebook or
dataset and pass the same `--run-id`.

`successful`, `review`, and `failed` are the three-way outcome; `failed` is the
runner's `not_generatable` classification. The exact per-prompt rule list is in
`summary.json`.
